In [1]:
import sys
!{sys.executable} -m pip install matplotlib
!{sys.executable} -m pip install google-generativeai google.api_core

  Using cached google_generativeai-0.8.6-py3-none-any.whl.metadata (3.9 kB)
  Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata (5.7 kB)
  Using cached proto_plus-1.27.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached grpcio_status-1.80.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached grpcio_status-1.78.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached grpcio_status-1.76.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.75.1-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.75.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.74.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.73.1-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.73.0-py3-none-any.whl

In [2]:
# import json
# from openai import OpenAI, RateLimitError
# from dotenv import load_dotenv
# import time
# import os

# load_dotenv('/Users/yvesb/Documents/llm-agents-gama/llm-agents/.envrc')  # This reads the .env file in the current directory


# model = "openai/gpt-oss-120b"
# client = OpenAI(
#     base_url="https://api.groq.com/openai/v1",
#     api_key=os.getenv("GROQ_API_KEY"),
# )

# def prompt_Groq(system_prompt,user_prompt, max_retries=3, delay=30, temperature=0.7, max_tokens=3000):
#     retries = 0
#     start_time = time.time()
#     while retries < max_retries:
#         try:
#             rep = client.chat.completions.create(
#                 model=model,  # or your preferred model
#                 messages=[
#                     {"role": "system", "content": system_prompt},
#                     {"role": "user", "content": user_prompt},
#                 ],
#                 temperature=temperature,
#                 max_tokens=max_tokens,
#             )
#             print(f"LLM[ {model}, {temperature} ] SUCCESS: Response received in {round(time.time() - start_time, 2)} seconds. token used: {rep.usage.total_tokens}, retry: {retries}")

#             return rep
#         except RateLimitError as e:
#             retries += 1
#             wait_time = max(int(e.response.headers.get('Retry-After', delay)), 3*60)  # Temps d'attente suggéré (max 3 minutes) ou 30s par défaut
#             print(e)  # Log de l'erreur complète pour diagnostic
#             print(f"LLM[ {model}, {temperature} ]: Rate limit exceeded. Retry {retries}/{max_retries}. Waiting {wait_time} seconds...")
#             time.sleep(wait_time)
#         except Exception as e:
#             print(f"LLM[ {model}, {temperature} ]: Unexpected error: {e}")
#             raise  # Relance l'erreur si ce n'est pas une limite de taux

In [6]:
import os
import time
import google.generativeai as genai
from google.api_core import exceptions
from dotenv import load_dotenv

# Chargement de l'environnement
load_dotenv('/Users/yvesb/Documents/llm-agents-gama/llm-agents/.envrc')

# Configuration de l'API Gemini
genai.configure(api_key=os.getenv("PROVIDER_KEYS__google"))

# Sélection du modèle
#MODEL_NAME = "gemini-2.5-flash-lite"
MODEL_NAME = "gemini-2.5-flash"
#MODEL_NAME = "gemini-3.1-flash-lite-preview" # => TOP

for m in genai.list_models():
    print(m.name)
    #print(m.input_token_limit)
    #print(m.description)

def prompt(system_prompt, user_prompt, max_retries=3, delay=60, temperature=0.7, max_tokens=3000):
    """
    Interroge Gemini 1.5 Flash avec gestion du Free Tier (15 RPM).
    """
    # Configuration du modèle avec le System Instruction
    model = genai.GenerativeModel(
        model_name=MODEL_NAME,
        system_instruction=system_prompt,
        generation_config={
            "temperature": temperature,
            "max_output_tokens": max_tokens,
        }
    )

    retries = 0
    start_time = time.time()

    while retries < max_retries:
        try:
            # Envoi de la requête
            response = model.generate_content(user_prompt)
            
            # Calcul du temps et log technique
            duration = round(time.time() - start_time, 2)
            # Note : Le SDK Google ne renvoie pas l'usage exact de la même façon que OpenAI
            print(f"LLM[ {MODEL_NAME}, {temperature} ] SUCCESS: Received in {duration}s. Retry: {retries}")
            return response

        except exceptions.ResourceExhausted as e:
            # Erreur 429 : Quota dépassé (15 RPM ou 1M TPM)
            retries += 1
            print(f"LLM[ {MODEL_NAME} ]: Rate limit exceeded. Retry {retries}/{max_retries}. Waiting {delay}s...")
            time.sleep(delay)
            
        except Exception as e:
            print(f"LLM[ {MODEL_NAME} ]: Unexpected error: {e}")
            raise


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [3]:
transportList = [
    "Walking",
    "Bike",
    "Bus / Metro",
    "Train", 
    "Private Car"
]

satisfactionScale = [
    (0.0, "Horrible"),
    (0.1, "Terrible"),
    (0.2, "Poor"),
    (0.3, "Unsatisfactory"),
    (0.4, "Subpar"),
    (0.5, "Average"),
    (0.6, "Fair"),
    (0.7, "Good"),
    (0.8, "Very Good"),
    (0.9, "Excellent"),
    (1.0, "Perfect")
]

getSatisfactionLabel = lambda value: next(label for threshold, label in satisfactionScale if value <= threshold)

#1 INIT

/Users/yvesb/Documents/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/sd/yfpxc_y13hqfry2rspfkspq80000gr/T/ipykernel_41110/3871646583.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/gemini-embedding-2-preview
models/aqa
models/imagen-4.0-generate-001
models/imagen-4.0-ultra-generate-001
models/imagen-4.0-fast-generate-001
models/veo-2.0-generate-001
models/veo-3.0-generate-001
models/veo-3.0-fast-generate-001
models/veo-3.1-generate-preview
models/veo-3.1-fast-generate-preview
models/veo-3.1-lite-generate-preview
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview

In [6]:
import numpy as np
import copy

CRITERION = ["Cost", "Time", "Ease", "Safety", "Comfort", "Ecology"]

class Persona:

    _id_counter = 1

    def __init__(self, name, profile, primary_goal, organization, constraints, sensitivities):
        self.id = self._id_counter
        self._id_counter += 1
        self.name = name
        self.profile = profile
        self.primary_goal = primary_goal
        self.organization = organization
        self.constraints = constraints
        self.sensitivities = [{}]
        for mode in transportList:
            self.sensitivities[0][mode] = {criterion: 0.5 for criterion in CRITERION}
        
        for mode in transportList:
            self.update_sensitivity(mode, sensitivities[mode], new_line=False, update_transport_feeling=mode==transportList[-1])



    def update_sensitivity(self, mode, mode_sensitivities, new_line, update_transport_feeling):
        if new_line:
            for mode in transportList:
                self.sensitivities.append(copy.deepcopy(self.sensitivities[-1]))

        for criterion in CRITERION:
            self.sensitivities[-1][mode][criterion] = mode_sensitivities[criterion]

        if update_transport_feeling:
            self.set_transportation_feeling()
            self.order_travel_mode()

    def get_current_sensitivity(self, mode=None):
        if mode:
            return copy.deepcopy(self.sensitivities[-1][mode])
        return copy.deepcopy(self.sensitivities[-1])

    def reset_sensitivity(self):
        self.sensitivities = [self.sensitivities[0].copy()]


    def set_transportation_feeling(self):
        feeling = "My sensitivities about transportation modes are as follows:\n"
        for mode, criteria in self.sensitivities[-1].items():
            scores = list(criteria.values())

            if not scores:  # Gestion des cas vides (optionnel mais robuste)
                feeling += f"Mode {mode}: no sensitivities defined.\n"
                continue

            mean_criteria = round(np.mean(scores), 2)
            details = ", ".join(
                f"{criterion}: {getSatisfactionLabel(value)}"
                for criterion, value in criteria.items()
            )
            feeling += f"Mode {mode}: global sensitivity is {getSatisfactionLabel(mean_criteria)} => {details}\n"

        self.transportation_feeling = self.summarize_transportation_experience(feeling)
        print(f"Updated transportation feeling for {self.name}:\n{self.transportation_feeling}\n")

    def get_transportation_feeling(self):
        return self.transportation_feeling
    
    def order_travel_mode(self):
        print("Ordering travel modes based on current sensitivities...")
        self.ordered_mode = []
        for mode, criteria in self.get_current_sensitivity().items():
            avg_score = np.mean(list(criteria.values()))
            self.ordered_mode.append((mode, avg_score))
        self.ordered_mode.sort(key=lambda x: x[1], reverse=True)


    def summarize_transportation_experience(self,feeling):
        print("Summary of transportation experience...")
        system_prompt = """
            Transform the given persona data into a **short, natural first-person story** about their daily transportation experiences.
            Use simple, direct language. 
            Rules:
            - Focus 80% of the story on **why** they prefer or dislike specific modes based on the sensitivity scores (Ease, Comfort, Safety, etc.) and on experience
            - Treat Goal, Profile, and Routine as brief context only.
            - Use natural, descriptive language instead of technical terms (e.g., instead of "Comfort: 1.0", say "I need that premium, quiet cabin experience").
            - Keep it to 5-6 concise sentences.
            - Maintain a realistic, "interview-style" first-person tone.
            """

        user_prompt = f"""
        Persona details:
        - Profile: {self.profile}
        - Goal: {self.primary_goal}
        - Routine: {self.organization}
        - Constraints: {self.constraints}
        - Sensitivities: {feeling}

        Experience:
        - Smooth commute: Exactly 18 min, music cranked up, arrived relaxed and on time.
        - Carpooling with a colleague: We chatted the entire way; I didn't even notice the time passing despite the traffic.
        - Shared carpool: It’s really the ideal way to cut costs and my carbon footprint while staying comfortable.
        - Smooth commute: 18 min, AC on, arrived fresh and rested.
        - bike ride: 22 min on a wide, protected cycle path; felt safe, disconnected from the traffic noise.
        - Standard traffic jam: 30 min, podcast to pass the time, arrived 5 min late.


        Example style:
        "I'm [age], [job] in [location]. [Mode 1] is [like/dislike] because [reason], but [downside]. [Mode 2] is [like/dislike] for [reason], but [issue]. Due to [constraints], I mostly rely on [preferred mode] despite [drawback]."
        """
        print(f"System Prompt:\n{system_prompt}\n")
        print(f"User Prompt:\n{user_prompt}\n")
        response = prompt(system_prompt, user_prompt)
        if MODEL_NAME.startswith("gemini"):
            return response.text
        return response.choices[0].message.content

    def __str__(self):
        feeling_lines = ". \n".join(self.transportation_feeling.split(". "))

        info = (
            f"Persona {self.id}: {self.name}\n"
            f"Profile: {self.profile}\n"
            f"Primary goal: {self.primary_goal}\n"
            f"Constraints: {self.constraints}\n"
            f"Organization: {self.organization}\n\n"
            f"Transportation Feeling:\n{feeling_lines}\n\n"
            f"Ordered Travel Modes (by preference):\n"
        )

            # Affichage de l'historique complet des sensitivities
        info += "Historique des sensitivities par moyen de transport:\n"
        for i, sensitivity_step in enumerate(self.sensitivities):
            info += f"\n--- Étape {i + 1} ---\n"
            for mode, criteria in sensitivity_step.items():
                info += f"\nMode: {mode}\n"
                for criterion, value in criteria.items():
                    info += f"  - {criterion}: {value:.2f}\n"

        for mode, score in self.ordered_mode:
            info += f"- {mode}: {round(score, 2)}\n"
        return info



In [ ]:

marc = Persona(
    name="The Multimodal Commuter (Marc)",
    profile="38y, Aeronautical Engineer, Pibrac.",
    primary_goal="Travel time reliability to avoid unpredictable traffic congestion.",
    organization="Daily home-work commute to city center; car used only for weekend leisure.",
    constraints="Strict dependence on SNCF schedules and connections at Matabiau station.",
    sensitivities={
            "Private Car": {"Cost": 0.6, "Time": 0.3, "Ease": 0.5, "Safety": 0.9, "Comfort": 0.9, "Ecology": 0.4},
            "Walking": {"Cost": 1.0, "Time": 0.8, "Ease": 1.0, "Safety": 0.8, "Comfort": 0.7, "Ecology": 1.0},
            "Train": {"Cost": 0.8, "Time": 0.9, "Ease": 0.8, "Safety": 0.9, "Comfort": 0.8, "Ecology": 0.9},
            "Bus / Metro": {"Cost": 0.8, "Time": 0.7, "Ease": 0.7, "Safety": 0.8, "Comfort": 0.6, "Ecology": 0.9},
            "Bike": {"Cost": 0.8, "Time": 0.7, "Ease": 0.8, "Safety": 0.6, "Comfort": 0.7, "Ecology": 0.9}
        }
)

sarah = Persona(
    name="The Budget Student (Sarah)",
    profile="21y, Master's student, Borderouge.",
    primary_goal="Radical minimization of travel costs.",
    organization="Trips to Rangueil campus and Jean Jaurès; grocery shopping on foot.",
    constraints="No personal vehicle or license; budget limited to Tisséo social fares.",
    sensitivities={
            "Private Car": {"Cost": 0.1, "Time": 0.5, "Ease": 0.2, "Safety": 0.8, "Comfort": 0.8, "Ecology": 0.5},
            "Walking": {"Cost": 1.0, "Time": 0.6, "Ease": 0.9, "Safety": 0.7, "Comfort": 0.6, "Ecology": 1.0},
            "Train": {"Cost": 0.8, "Time": 0.9, "Ease": 0.8, "Safety": 0.9, "Comfort": 0.8, "Ecology": 0.9},
            "Bike": {"Cost": 1.0, "Time": 0.8, "Ease": 0.9, "Safety": 0.5, "Comfort": 0.5, "Ecology": 1.0},
            "Bus / Metro": {"Cost": 1.0, "Time": 0.9, "Ease": 1.0, "Safety": 0.8, "Comfort": 0.7, "Ecology": 0.9}     
        }
    )

thomas = Persona(
    name="Thomas, the Wealth Manager (Status & Performance)",
    profile="38y, Private Banker, lives in Vieille-Toulouse.",
    primary_goal="Reflect professional success and enjoy a high-end sensory experience during commutes.",
    organization="Primarily a direct commute between home and the city center or Labège. The car remains in an underground parking for 9 hours daily, with occasional short trips within the city center during the week for business lunches or high-profile client meetings.",
    constraints="None",
    sensitivities={
            "Private Car": {"Cost": 0.3, "Time": 0.9, "Ease": 1.0, "Safety": 0.9, "Comfort": 1.0, "Ecology": 0.4},
            "Walking": {"Cost": 1.0, "Time": 0.1, "Ease": 0.2, "Safety": 0.8, "Comfort": 0.4, "Ecology": 1.0},
            "Train": {"Cost": 0.8, "Time": 0.2, "Ease": 0.0, "Safety": 0.9, "Comfort": 0.5, "Ecology": 0.9},
            "Bike": {"Cost": 0.9, "Time": 0.3, "Ease": 0.1, "Safety": 0.2, "Comfort": 0.4, "Ecology": 0.9},
            "Bus / Metro": {"Cost": 0.8, "Time": 0.2, "Ease": 0.3, "Safety": 0.7, "Comfort": 0.5, "Ecology": 0.9}
        }
)


lea = Persona(
    name="X",
    profile="People",
    primary_goal="Daily commute.",
    organization="Daily commute.",
    constraints="None",
    sensitivities={
            "Private Car": {"Cost": 0.2, "Time": 0.3, "Ease": 0.3, "Safety": 0.9, "Comfort": 0.9, "Ecology": 0.1},
            "Walking": {"Cost": 1.0, "Time": 0.5, "Ease": 0.8, "Safety": 0.9, "Comfort": 0.7, "Ecology": 1.0},
            "Train": {"Cost": 0.8, "Time": 0.4, "Ease": 0.6, "Safety": 0.9, "Comfort": 0.8, "Ecology": 0.9},
            "Bike": {"Cost": 0.8, "Time": 1.0, "Ease": 0.9, "Safety": 0.7, "Comfort": 0.8, "Ecology": 1.0},
            "Bus / Metro": {"Cost": 0.8, "Time": 0.5, "Ease": 0.4, "Safety": 0.8, "Comfort": 0.6, "Ecology": 0.9}
        }
    )

jean = Persona(
    name="The Local Senior (Jean)",
    profile="70y, Retired, Les Minimes.",
    primary_goal="Maintaining social ties and simplified access to healthcare services.",
    organization="Daily trips for market, community activities, and medical appointments.",
    constraints="Physical fatigue; requirement for constant PRM accessibility.",
    sensitivities={
            "Private Car": {"Cost": 0.5, "Time": 0.6, "Ease": 0.4, "Safety": 0.8, "Comfort": 0.8, "Ecology": 0.5},
            "Walking": {"Cost": 1.0, "Time": 0.4, "Ease": 0.5, "Safety": 0.7, "Comfort": 0.4, "Ecology": 1.0},
            "Train": {"Cost": 0.8, "Time": 0.8, "Ease": 0.5, "Safety": 0.7, "Comfort": 0.6, "Ecology": 0.7},
            "Bike": {"Cost": 0.9, "Time": 0.4, "Ease": 0.2, "Safety": 0.3, "Comfort": 0.3, "Ecology": 0.9},
            "Bus / Metro": {"Cost": 1.0, "Time": 0.8, "Ease": 0.9, "Safety": 0.9, "Comfort": 0.8, "Ecology": 0.9}
        }
    )



    


persona_sensitivities = [marc,sarah,thomas,lea,jean]

In [11]:
cadre_dynamique = Persona(
    name="The Dynamic Executive",
    profile="50y, Senior Executive, polished appearance.",
    primary_goal="Reflecting professional success and maintaining personal discipline.",
    organization="Direct commutes between home and office (City Center or Labège).",
    constraints="Requirement for punctuality and high-end sensory standards.",
    sensitivities={
        "Private Car": {"Cost": 0.8, "Time": 0.9, "Ease": 1.0, "Safety": 1.0, "Comfort": 1.0, "Ecology": 0.7},
        "Walking": {"Cost": 1.0, "Time": 0.5, "Ease": 0.7, "Safety": 0.8, "Comfort": 0.5, "Ecology": 1.0},
        "Bike": {"Cost": 0.9, "Time": 0.6, "Ease": 0.4, "Safety": 0.7, "Comfort": 0.6, "Ecology": 0.9},
        "Bus / Metro": {"Cost": 0.9, "Time": 0.6, "Ease": 0.5, "Safety": 0.6, "Comfort": 0.3, "Ecology": 0.9},
        "Train": {"Cost": 0.8, "Time": 0.9, "Ease": 0.7, "Safety": 0.5, "Comfort": 0.6, "Ecology": 0.8}
    }
)

etudiante = Persona(
    name="The Pragmatic Student",
    profile="24y, Master's Student, athletic profile.",
    primary_goal="Optimizing time/cost ratio and maintaining physical activity.",
    organization="Commutes for university, sports, and social activities.",
    constraints="Limited budget; seeking maximum independence.",
    sensitivities={
        "Bike": {"Cost": 1.0, "Time": 1.0, "Ease": 0.9, "Safety": 0.7, "Comfort": 0.7, "Ecology": 1.0},
        "Walking": {"Cost": 1.0, "Time": 0.4, "Ease": 1.0, "Safety": 0.8, "Comfort": 0.8, "Ecology": 1.0},
        "Train": {"Cost": 0.8, "Time": 0.9, "Ease": 0.7, "Safety": 0.5, "Comfort": 0.6, "Ecology": 0.8},
        "Bus / Metro": {"Cost": 0.9, "Time": 0.6, "Ease": 0.7, "Safety": 0.7, "Comfort": 0.5, "Ecology": 0.9},
        "Private Car": {"Cost": 0.2, "Time": 0.4, "Ease": 0.3, "Safety": 0.9, "Comfort": 0.8, "Ecology": 0.3}
    }
)

Summary of transportation experience...
System Prompt:

            Transform the given persona data into a **short, natural first-person story** about their daily transportation experiences.
            Use simple, direct language. 
            Rules:
            - Focus 80% of the story on **why** they prefer or dislike specific modes based on the sensitivity scores (Ease, Comfort, Safety, etc.) and on experience
            - Treat Goal, Profile, and Routine as brief context only.
            - Use natural, descriptive language instead of technical terms (e.g., instead of "Comfort: 1.0", say "I need that premium, quiet cabin experience").
            - Keep it to 5-6 concise sentences.
            - Maintain a realistic, "interview-style" first-person tone.
            

User Prompt:

        Persona details:
        - Profile: 50y, Senior Executive, polished appearance.
        - Goal: Reflecting professional success and maintaining personal discipline.
        - Routine: Direct co

In [ ]:
cadre_dynamique = Persona(
    name="The Dynamic Executive",
    profile="50y, Senior Executive, polished appearance.",
    primary_goal="Reflecting professional success and maintaining personal discipline.",
    organization="Direct commutes between home and office (City Center or Labège).",
    constraints="Requirement for punctuality and high-end sensory standards.",
    sensitivities={
        "Private Car": {"Cost": 0.4, "Time": 0.8, "Ease": 1.0, "Safety": 0.2, "Comfort": 0.5, "Ecology": 0.7},
        "Walking": {"Cost": 1.0, "Time": 0.6, "Ease": 0.7, "Safety": 0.8, "Comfort": 0.5, "Ecology": 1.0},
        "Bike": {"Cost": 0.9, "Time": 0.6, "Ease": 0.5, "Safety": 0.7, "Comfort": 0.6, "Ecology": 0.9},
        "Bus / Metro": {"Cost": 0.9, "Time": 0.6, "Ease": 0.5, "Safety": 0.6, "Comfort": 0.4, "Ecology": 0.9},
        "Train": {"Cost": 0.8, "Time": 0.9, "Ease": 0.7, "Safety": 0.5, "Comfort": 0.6, "Ecology": 0.8}
    }
)



Summary of transportation experience...
System Prompt:

            Transform the given persona data into a **short, natural first-person story** about their daily transportation experiences.
            Use simple, direct language. 
            Rules:
            - Focus 80% of the story on **why** they prefer or dislike specific modes based on the sensitivity scores (Ease, Comfort, Safety, etc.) and on experience
            - Treat Goal, Profile, and Routine as brief context only.
            - Use natural, descriptive language instead of technical terms (e.g., instead of "Comfort: 1.0", say "I need that premium, quiet cabin experience").
            - Keep it to 5-6 concise sentences.
            - Maintain a realistic, "interview-style" first-person tone.
            

User Prompt:

        Persona details:
        - Profile: 50y, Senior Executive, polished appearance.
        - Goal: Reflecting professional success and maintaining personal discipline.
        - Routine: Direct co

In [7]:
etudiante = Persona(
    name="The Pragmatic Student",
    profile="24y, Master's Student, athletic profile.",
    primary_goal="Optimizing time/cost ratio and maintaining physical activity.",
    organization="Commutes for university, sports, and social activities.",
    constraints="Limited budget; seeking maximum independence.",
    sensitivities={
        "Bike": {"Cost": 0.8, "Time": 1.0, "Ease": 0.9, "Safety": 0.7, "Comfort": 0.6, "Ecology": 1.0},
        "Walking": {"Cost": 1.0, "Time": 0.4, "Ease": 1.0, "Safety": 0.8, "Comfort": 0.6, "Ecology": 1.0},
        "Train": {"Cost": 0.7, "Time": 0.9, "Ease": 0.7, "Safety": 0.5, "Comfort": 0.6, "Ecology": 0.8},
        "Bus / Metro": {"Cost": 0.8, "Time": 0.6, "Ease": 0.7, "Safety": 0.7, "Comfort": 0.5, "Ecology": 0.9},
        "Private Car": {"Cost": 0.6, "Time": 0.5, "Ease": 0.4, "Safety": 0.9, "Comfort": 1.0, "Ecology": 0.7}
    }
)



Summary of transportation experience...
System Prompt:

            Transform the given persona data into a **short, natural first-person story** about their daily transportation experiences.
            Use simple, direct language. 
            Rules:
            - Focus 80% of the story on **why** they prefer or dislike specific modes based on the sensitivity scores (Ease, Comfort, Safety, etc.) and on experience
            - Treat Goal, Profile, and Routine as brief context only.
            - Use natural, descriptive language instead of technical terms (e.g., instead of "Comfort: 1.0", say "I need that premium, quiet cabin experience").
            - Keep it to 5-6 concise sentences.
            - Maintain a realistic, "interview-style" first-person tone.
            

User Prompt:

        Persona details:
        - Profile: 24y, Master's Student, athletic profile.
        - Goal: Optimizing time/cost ratio and maintaining physical activity.
        - Routine: Commutes for univer

In [ ]:
print(jean)



In [ ]:
trajectList = [
    (0, "Train", "21 min (20 km) · Every 30 min"),
    (1, "Bus / Metro", "19 min · Every 3 min"),
    (2, "Private Car", "25 min (18.4 km) via A624 and A620"),
    (3, "Bike", "6 min (1.5 km) via Pont Neuf"),
    (4, "Bus / Metro", "12 min · Every 10 min"),
    (5, "Bus / Metro", "10 min · Every 5 min"),
    (6, "Bus / Metro", "15 min · Transfer at Jean Jaurès"),
    (7, "Walking", "50 min · flat, in green parks"),
    (8, "Walking", "30 min · Very sporty with a lot of road to cross"),
    (9, "Private Car", "18 min (12.2 km) via A62"),
    (10, "Bike", "10 min (2.1 km) via Bd de Strasbourg"),
    (11, "Train", "25 min · Transfer at Saint-Agne SNCF"),
    (12, "Bus / Metro", "30 min · Every 10 min"),
    (13, "Walking", "40 min (1800 m) via Rue de la Colombette, a lot of elevation gain"),
    (14, "Bus / Metro", "14 min · Every 3 min"),
    (15, "Bus / Metro", "15 min · Every 12 min"),
    (16, "Private Car", "22 min (15.1 km) via A61"),
    (17, "Bike", "8 min (2.0 km) via Canal du Midi"),
    (18, "Bus / Metro", "20 min · Every 15 min")
]

In [ ]:
def get_best_traject_llm(traveller, trajectList,temperature=0.7):

    system_prompt = """
    You are a transportation expert. Your task is to recommend the best transportation mode for the given persona based on their profile, goals, constraints, and sensitivities.

    **Instructions:**
    - Analyze the persona's preferences, constraints, and the trajectory options.
    - Select the **single best option** (by index) that aligns with their needs.
    - Return your answer in **JSON format** with the following keys:
      - "chosen_index": int (index of the best trajectory from the list)
      - "mode": str (transportation mode)
      - "reason": str (brief justification, max 30 words)

    **Example Output:**
    {
      "chosen_index": 3,
      "mode": "Electric Bike",
      "reason": "Fast, eco-friendly, and aligns with their active lifestyle and short distance."
    }
    """

    user_prompt = f"""
      Persona details:
      {{
        "sensitivities": "{traveller.transportation_feeling}"
      }}

    Trajectory options (index, mode, description):
    {json.dumps(trajectList, indent=2)}

    Provide your recommendation as valid JSON only, no additional text.
    """

    response = prompt(system_prompt, user_prompt, temperature=temperature)
    # print(f"LLM Response: [{response.text}]")  # Log de la réponse brute pour diagnostic
    # print(f"LLM Response replace: [{response.text.replace('json', '')}]")  # Log de la réponse brute pour diagnostic

    if MODEL_NAME.startswith("gemini"):
        return json.loads(response.text.replace("json", ""))
    return json.loads(response.choices[0].message.content)


In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

def plot_recommendations(res_stat, trajectList, persona_name):
    indices = list(res_stat.keys())
    counts = list(res_stat.values())
    modes = [trajectList[i][1] for i in indices]

    plt.figure(figsize=(12, 6))
    bars = plt.bar(modes, counts, color='skyblue')

    plt.title(f"Recommandation Frequency for {persona_name}")
    plt.xlabel("Transportation Mode")
    plt.ylabel("Number of Recommendations")
    plt.xticks(rotation=45, ha='right')

    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2., height,
                 f'{int(height)}',
                 ha='center', va='bottom')

    plt.tight_layout()
    plt.show()


In [ ]:
import csv
from collections import defaultdict

def save_to_csv(filename, temperature, iteration, persona, res_stat):
    with open(filename, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)

        # Écrire l'en-tête si le fichier est vide
        if file.tell() == 0:
            header = ["temperature", "Persona","Narative","Iteration", "Ordered Mode"]

            # Ajouter les colonnes nb_0 à nb_18 (si nécessaires)
            header.extend([f"nb_{i}" for i in range(19)])

            # Ajouter les colonnes pour les valeurs de sensibilité (par mode de transport)
            transport_modes = persona.sensitivities[-1].keys()  # Ex: ["Walking", "Bike", ...]
            header.extend([f"{mode} Values" for mode in transport_modes])

            writer.writerow(header)

        # Préparer la ligne de données
        row = [
            temperature,
            persona.name,
            persona.transportation_feeling,
            iteration,
            " > ".join([mode for mode, _ in persona.ordered_mode])  # Ex: "Private Car > Train > Bike"
        ]

        # Ajouter les statistiques (nb_0 à nb_18)
        for i in range(19):
            row.append(res_stat.get(i, 0))

        # Ajouter les valeurs de sensibilité (format JSON propre)
        for mode, criteria in persona.sensitivities[-1].items():
            row.append(json.dumps(criteria, ensure_ascii=False))  # Ex: '{"Cost": 0.4, "Time": 0.8, ...}'

        writer.writerow(row)

In [ ]:
import random
max_history = 20
traveller = lea
traveller.reset_sensitivity() 

for it in range(10):
    print(f"\n------------------\nTesting for persona: {traveller.name} - Iteration {it + 1}")

    # Reset statistics for this iteration
    res_stat = defaultdict(int)

    for transport in transportList:
        direction = random.choice([-1, 1])  # Hausse (1) ou Baisse (-1)
        power = random.uniform(0.5, 1.5)

        update_car_sens = traveller.get_current_sensitivity(transport).copy()
        if direction == -1:
            update_car_sens = {criterion: round(max(value - (value-0.2)/(10-it)*power, 0.0), 2) for criterion, value in update_car_sens.items()}
        else:
            update_car_sens = {criterion: round(min(value + (1.0-value)/(10-it)*power, 1.0), 2) for criterion, value in update_car_sens.items()}
        
        traveller.update_sensitivity(transport, update_car_sens, new_line=True, update_transport_feeling=False)

    # Get recommendations from LLM (10 runs)
    temperature = 0.7
    for run in range(10):
        json_response = get_best_traject_llm(traveller, trajectList,temperature)
        print(json_response)  # Log des réponses dans le notebook
        res_stat[json_response['chosen_index']] += 1

    # Sauvegarder les résultats
    save_to_csv(
        'transport_recommendations.csv',
        temperature,
        it + 1,
        traveller,
        res_stat
    )


    print("Statistics:", dict(res_stat))

Itération 1:
J'ai 38 ans, je suis banquier privé et je vis à Vieille-Toulouse. Je souhaite que mes trajets domicile-travail soient synonymes de réussite et me procurent une expérience haut de gamme et sensorielle. Je pourrais marcher : c'est gratuit et sûr, mais c'est interminable et inconfortable. Le vélo est également bon marché, mais dangereux et lent. Le métro et le train sont économiques et écologiques, mais les trajets sont longs, le service peu pratique et le confort moyen. La voiture particulière, en revanche, m'offre rapidité, confort et le prestige que j'apprécie, même si c'est coûteux et pas l'option la plus écologique. Je laisse donc ma voiture au parking souterrain pendant neuf heures et je me rends directement au centre-ville ou à Labège, ne sortant que pour un déjeuner d'affaires rapide.

Itération 2:
J'ai 38 ans, je suis banquier privé et je vis à Vieille-Toulouse. Mes journées sont rythmées par la volonté de projeter une image de réussite professionnelle et de savourer une expérience sensorielle haut de gamme lors de mes trajets vers le centre-ville ou Labège. J'adore conduire ma voiture : c'est rapide, confortable et luxueux, même si le coût du stationnement et son impact environnemental sont un vrai problème. Marcher ou faire du vélo serait économique et écologique, mais le temps que cela prend et le manque de confort rendent ces options peu pratiques pour mon emploi du temps. Le métro est abordable et écologique, mais les rames bondées et la lenteur du service ne correspondent jamais au niveau de raffinement que j'attends. Sans véritable contrainte, je privilégie ma voiture, acceptant son coût et son impact écologique négatif car elle me procure l'expérience dont j'ai besoin.

Itération 3:
J'ai 38 ans et je travaille comme banquier privé à Vieille-Toulouse. Ma journée est rythmée par la réussite professionnelle et le plaisir d'un trajet domicile-travail haut de gamme, offrant une véritable expérience sensorielle, entre mon domicile et le centre-ville ou Labège. Je pourrais y aller à pied : c'est gratuit et sûr, mais le temps de trajet est interminable et inconfortable. Le vélo est économique et rapide, mais la conduite est peu pratique et la sécurité moyenne. Le bus, le métro ou le train sont bon marché et écologiques, mais lents, difficiles à utiliser et jamais synonymes de luxe. Je privilégie donc ma voiture : rapide, très pratique et offrant une sensation de luxe, même si elle pèse sur mon budget et l'environnement.

Itération 4:
J'ai 38 ans, je suis banquier privé et je vis à Vieille-Toulouse. Mes journées sont rythmées par la volonté de projeter une image de réussite professionnelle et de profiter d'une expérience sensorielle haut de gamme lors de mes trajets domicile-travail. Je conduis ma voiture personnelle car elle est confortable, sûre et rapide, même si elle coûte cher et n'est pas très écologique. Marcher me semble sûr et écologique, mais c'est terriblement lent et inconfortable pour mon emploi du temps. Le vélo est économique et écologique, mais le confort de conduite n'est pas à la hauteur de l'image que je souhaite projeter. Le bus, le métro et le train sont bon marché et écologiques, mais ils sont lents, bondés et peu pratiques ; je privilégie donc ma voiture malgré son prix.

Itération 5:
J'ai 38 ans, je suis banquier privé et je vis à Vieille-Toulouse. Mes journées sont rythmées par l'affichage de ma réussite professionnelle et le plaisir d'un trajet domicile-travail haut de gamme entre mon domicile et le centre-ville ou Labège. J'adore ma voiture : elle est élégante, confortable et me permet d'arriver avec style, même si elle est coûteuse et peu écologique. Marcher est économique et sûr, mais c'est long et inconfortable, alors je l'évite. Le vélo est une alternative correcte, économique et rapide, mais le confort n'est pas optimal. Sans contraintes particulières, j'utilise ma voiture pour mes longs trajets quotidiens et je ne prends le vélo ou ne marche que pour de courts déplacements en ville, malgré leurs inconvénients.

Itération 6:
J'ai 38 ans, je suis banquier privé et je vis à Vieille-Toulouse. Mes journées sont rythmées par l'affichage de ma réussite professionnelle et le plaisir d'un trajet domicile-travail haut de gamme et sensoriel. J'adore ma voiture car elle est confortable et sécurisante, mais son coût et son impact environnemental sont loin d'être idéaux. Marcher est économique et écologique, mais terriblement lent et inconfortable pour moi. Le vélo est fluide, économique et écologique, mais je l'utilise rarement lorsque je souhaite projeter une image prestigieuse. Sans véritable contrainte, je laisse tout de même ma voiture au parking souterrain pendant neuf heures et je compte sur elle pour mes principaux trajets domicile-ville, me contentant de sa durée moyenne et de son confort.

Itération 7:
J'ai 38 ans, je suis banquier privé et je vis à Vieille-Toulouse. Mes journées sont rythmées par la réussite professionnelle et des trajets domicile-travail haut de gamme, sources de plaisir sensoriel. J'adore mon vélo : il est économique, rapide, très confortable et écologique, mais il ne me donne pas l'image de cadre que je souhaite projeter lors de mes déjeuners d'affaires. Ma voiture, quant à elle, offre un confort, un rapport qualité-prix et un impact environnemental moyens, mais elle projette le statut social recherché et est pratique pour les courts trajets en centre-ville. J'évite de marcher : c'est lent et inconfortable. Quant au bus, au métro et au train, je les trouve trop longs et peu pratiques. Je me fie donc principalement à ma voiture pour mes déplacements quotidiens, même si elle n'est pas idéale.

Itération 8:
J'ai 38 ans, je suis banquier privé et je vis à Vieille-Toulouse. Mes journées sont consacrées à projeter une image de réussite et à profiter d'un trajet domicile-travail haut de gamme, offrant un véritable moment de détente et de bien-être, entre mon domicile et le centre-ville ou Labège. J'adore le vélo : c'est économique, rapide, confortable, sûr et totalement écologique, mais cela ne correspond pas à l'image soignée que je dois projeter lors de mes rendez-vous clients. Le métro que j'occupe pendant neuf heures me donne l'impression d'être un symbole de réussite, mais il est cher, lent, inconfortable et peu écologique. Il m'arrive de marcher ou de prendre le métro pour déjeuner rapidement ; la marche est sûre et économique, mais interminable, et le métro est abordable et écologique, mais exigu et peu fiable. Alors, malgré ses défauts, je privilégie la voiture pour conserver l'image prestigieuse que mes clients attendent.

Itération 9:
J'ai 38 ans, je suis banquier privé et je vis à Vieille-Toulouse. Mes journées sont rythmées par la recherche d'une image professionnelle réussie, tout en profitant d'un trajet domicile-travail raffiné. J'adore le vélo car il combine parfaitement coût, temps, facilité, sécurité, confort et écologie, même s'il n'est pas toujours pratique pour le long trajet jusqu'à Labège. Marcher est sûr et économique, mais terriblement lent et inconfortable, alors je le fais rarement. Le bus, le métro et le train sont abordables et écologiques, mais lents, bondés et difficiles d'accès. Malgré les piètres performances de ma voiture personnelle en termes de coût, temps, facilité, sécurité, confort et écologie, je la laisse au parking souterrain et je la conduis uniquement lorsque je souhaite projeter une image haut de gamme pour des déjeuners d'affaires.

Itération 10:
J'ai 38 ans, je suis banquier privé et je vis à Vieille-Toulouse. Mes journées sont rythmées par la volonté de projeter une image de réussite tout en profitant de trajets domicile-travail haut de gamme et sensoriels. J'adore le vélo car tous ses aspects – coût, rapidité, facilité, sécurité, confort et écologie – sont parfaits, même si cela peut paraître un peu informel pour des réunions importantes. La marche est pratique pour une petite balade, mais c'est interminable et inconfortable. Le bus, le métro et le train sont bon marché et écologiques, mais lents, bondés et difficiles à appréhender. Ma voiture est garée au parking souterrain neuf heures par jour, mais je l'utilise rarement car elle est chère, inconfortable et peu prestigieuse. Je privilégie donc mon vélo, même si j'ai parfois besoin d'un trajet plus formel.

Léa

Itération 1:
J'ai 34 ans, je suis médecin libérale à Saint-Cyprien et je jongle au quotidien entre un emploi du temps chargé et une activité physique régulière. J'adore le vélo : c'est économique, rapide, confortable et écologique, et cela me permet d'atteindre mon objectif de pas. Cependant, pour me sentir en sécurité, j'ai besoin de pistes cyclables bien aménagées et sécurisées, ainsi que d'une journée sèche. La marche est également une excellente option : c'est sûr, économique et écologique, même si c'est un peu plus long et moins confortable sur les trottoirs irréguliers. Le bus et le train sont une solution acceptable ; ils sont abordables et sûrs, mais le service est inégal et le confort des trajets est moyen. Lorsque le temps se gâte ou que je ne trouve pas d'endroit sûr pour garer mon vélo, je prends ma voiture, fiable et rapide, mais plus chère et moins écologique. La plupart du temps, je fais donc du vélo et ne marche ou ne prends la voiture que lorsque les conditions l'exigent.

Itération 2:
J'ai 34 ans et je suis médecin libérale à Saint-Cyprien. Ma journée est rythmée par les trajets domicile-travail, que je dois enchaîner avec un peu d'exercice. J'adore marcher : c'est économique, sûr et écologique, mais c'est trop long pour me rendre à mes rendez-vous. Le vélo est idéal en termes de temps et de coût, et j'apprécie l'activité physique, mais je m'inquiète de la pluie et du manque de pistes cyclables sécurisées. Le bus et le métro sont abordables et sûrs, mais je les trouve exigus et peu fiables, et le train est confortable, mais plus lent. Comme la météo est parfois capricieuse et que j'ai besoin d'un endroit sûr pour attacher mon vélo, je prends généralement ma voiture : c'est rapide et confortable, même si ce n'est pas l'option la plus écologique.

Itération 3:
J'ai 34 ans, je suis médecin libérale à Saint-Cyprien, et mes journées sont rythmées par l'optimisation de mon emploi du temps tout en essayant de faire un peu d'exercice. J'adore marcher car c'est économique, sûr et écologique, mais c'est plus long et la pluie me ralentit. Le vélo est idéal pour gagner du temps, économiser de l'argent et faire mon exercice quotidien, mais j'ai besoin de pistes cyclables sécurisées et c'est pénible quand il pleut. Les bus et le métro sont abordables et sûrs, mais peu pratiques et inconfortables, et les trains sont fiables mais plus lents. Comme je suis sensible aux conditions météorologiques et que j'ai besoin d'infrastructures cyclables de qualité, je fais surtout du vélo quand il fait beau et je prends la voiture sinon, même si cela signifie que je manque ma séance de vélo quotidienne.

Itération 4:
J'ai 34 ans et je suis médecin libérale à Saint-Cyprien. Mes journées sont rythmées par l'optimisation de mon emploi du temps tout en restant active. La marche est économique, sûre et écologique, mais un peu plus longue que je ne le souhaiterais. Le vélo est mon moyen de transport préféré : rapide, facile et encore peu coûteux, même s'il me faut de bonnes pistes cyclables et que la pluie complique les choses. Les bus et le métro sont abordables, mais souvent bondés et plus lents, et le train, bien que fiable, n'est pas aussi pratique pour mes courts trajets. Sensible aux aléas climatiques et ayant besoin d'infrastructures cyclables sécurisées, je privilégie mon vélo et ne prends la voiture que lorsque la pluie rend le vélo dangereux.

Itération 5:

Itération 6:

Itération 7:

Itération 8:

Itération 9:

Itération 10: